# Week 4 — Block 2: Guided Demo (Time-Series)

**DATS 6401 · Visualization of Complex Data**

~35 min on Mauna Loa CO₂ (ships with statsmodels):

1. Getting dates into pandas (~6 min)
2. Index, resample, resolution (~7 min)
3. ACF → the period → the matched rolling window (~8 min)
4. Decompose; read the residual; additive vs multiplicative (~8 min)
5. The ±2σ band + a bootstrap interval (~6 min)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

co2 = sm.datasets.co2.load_pandas().data.dropna()
print(type(co2.index), "| span:", co2.index.min().date(), "→", co2.index.max().date())
co2.head(3)

## Part 1 — Getting dates into pandas

CO₂ arrives with a DatetimeIndex already. Most data you load will not: dates
come in as text, and nothing time-aware works until they are converted.

In [ ]:
raw = pd.DataFrame({"when": ["2024-01-05", "2024-02-05", "2024-03-05"],
                    "ppm":  [421.0, 422.4, 423.1]})
print("before:", raw["when"].dtype)          # object -> just strings

raw["when"] = pd.to_datetime(raw["when"])
print("after: ", raw["when"].dtype)          # datetime64[ns]

# "03/04/2024" is 3 April in most of the world, 4 March in the US.
print(pd.to_datetime("03/04/2024").date())                 # pandas guesses
print(pd.to_datetime("03/04/2024", dayfirst=True).date())  # you decide
print(pd.to_datetime(["2024-01-05", "oops"], errors="coerce"))  # bad -> NaT

With the dates as the **index**, pandas understands the calendar. Note the last
pair: a row-count window and a time window are different things once the
spacing is uneven — this is a common source of silent bugs.

In [ ]:
d = pd.Series(range(90), index=pd.date_range("2024-01-01", periods=90, freq="D"))
print(d.loc["2024-02"].head(2))            # slice a whole month by name
print(d.resample("MS").mean().round(1))    # month-start averages
print(d.index.day_name()[:3].tolist())     # calendar parts

gappy = pd.Series([1, 2, 3, 4], index=pd.to_datetime(
    ["2024-01-01", "2024-01-02", "2024-01-10", "2024-01-11"]))
print("\n3 ROWS:\n", gappy.rolling(3).mean().round(2).tolist())
print("3 DAYS:\n", gappy.rolling("3D").mean().round(2).tolist())

## Part 2 — Resolution is a choice

A DatetimeIndex unlocks `resample` — and `.mean()` vs `.sum()` is itself a meaning choice (averaging CO₂ ✓; you'd SUM daily sales).

In [ ]:
monthly = co2["co2"].resample("MS").mean().dropna()
yearly  = co2["co2"].resample("YS").mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
co2["co2"].loc["1990":"1995"].plot(ax=axes[0], color="#2E6E8E")
axes[0].set_title("weekly slice: sawtooth dominates")
yearly.plot(ax=axes[1], color="#2E6E8E")
axes[1].set_title("yearly: only the trend survives")
plt.show()

## Part 3 — Find the period, then smooth with it

Don't guess the window: **ask the ACF (autocorrelation function).** Differencing
first — `.diff()` replaces each value with the change since the previous month —
strips the trend, so the seasonal spike is visible instead of being buried under
it.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(8, 3))
plot_acf(monthly.diff().dropna(), lags=36, ax=ax)
ax.set_title("ACF of monthly CHANGES: spikes at 12, 24, 36 → period = 12")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
seg = monthly.loc["1980":"1995"]
seg.plot(ax=ax, alpha=0.3, color="#5a6672", label="monthly")
seg.rolling(12).mean().plot(ax=ax, lw=2, color="#2E6E8E", label="w=12 (one full cycle)")
ax.legend(); ax.set_title("The matched window: seasonality cancels OUT of the trend")
plt.show()

Rather than describing what other windows would do, plot them. The right panel
rolls the **standard deviation** instead of the mean, which tracks how big the
seasonal swing is over time.

In [ ]:
seg = monthly.loc["1980":"1998"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
seg.plot(ax=axes[0], color="#bbbbbb", lw=0.9, label="monthly")
for w, c in [(3, "#d9534f"), (12, "#2E6E8E"), (60, "#6aa84f")]:
    seg.rolling(w).mean().plot(ax=axes[0], lw=1.8, color=c, label=f"w={w}")
axes[0].legend(fontsize=8)
axes[0].set_title("3 keeps the season · 12 removes it · 60 flattens real turns")

seg.rolling(12).std().plot(ax=axes[1], color="#b07aa1", lw=1.8)
axes[1].set_title("rolling std: size of the seasonal swing")
plt.tight_layout(); plt.show()

## Part 4 — Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

res = seasonal_decompose(monthly, period=12)
fig = res.plot(); fig.set_size_inches(8.5, 5.5)
plt.tight_layout(); plt.show()

**Read it bottom-up with the class:** the residual is close to structureless
noise, so the additive model fits this series. Now show what a series that needs
the *other* model looks like.

In [ ]:
# A series whose seasonal swing grows with its level.
rng2 = np.random.default_rng(11)
n = 132
grow = pd.Series(
    np.linspace(100, 320, n)
    * (1 + 0.16 * np.sin(2 * np.pi * np.arange(n) / 12))
    * (1 + rng2.normal(0, 0.012, n)),
    index=pd.date_range("2008-01-01", periods=n, freq="MS"))

add = seasonal_decompose(grow, period=12, model="additive")
mul = seasonal_decompose(grow, period=12, model="multiplicative")

fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
axes[0].plot(grow.index, grow.values, color="#2E6E8E")
axes[0].set_title("swing grows with the level")
axes[1].plot(add.resid.index, add.resid.values, color="#d9534f")
axes[1].axhline(0, color="#999", lw=0.7); axes[1].set_title("additive residual")
axes[2].plot(mul.resid.index, mul.resid.values, color="#6aa84f")
axes[2].axhline(1, color="#999", lw=0.7); axes[2].set_title("multiplicative residual")
plt.tight_layout(); plt.show()

**Point at the middle panel.** The additive residual is large at *both ends* and
near zero in the middle — not a growing wedge. `seasonal_decompose` estimates one
averaged seasonal shape, which is too big early and too small late, so it only
fits mid-series. The multiplicative residual has no pattern left.

The rule students should leave with: fit a model, then **read the residual
panel**. Leftover shape means the other model is the right one.

## Part 5 — Uncertainty, two ways

In [ ]:
roll = monthly.rolling(12)
mean, std = roll.mean(), roll.std()
fig, ax = plt.subplots(figsize=(9, 3.2))
mean.plot(ax=ax, color="#2E6E8E", label="12-mo mean")
ax.fill_between(mean.index, mean - 2*std, mean + 2*std, alpha=0.2, color="#2E6E8E", label="±2σ")
ax.legend(); ax.set_title("Band #1: data spread around the smooth line")
plt.show()

In [ ]:
# Band #2 — bootstrap CI for a STATISTIC (mean monthly increase in the 1990s)
rng = np.random.default_rng(0)
changes = monthly.diff().dropna().loc["1990":"1999"].values
boots = [rng.choice(changes, len(changes), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boots, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.hist(boots, bins=40, color="#2E6E8E", alpha=0.85)
for v in (lo, hi): ax.axvline(v, color="#d9534f", lw=2)
ax.set_title(f"Bootstrap: mean 1990s monthly rise, 95% CI [{lo:.3f}, {hi:.3f}] ppm")
plt.show()

**Narrate the distinction once more:** ±2σ answers "how far do values stray"; the CI answers "how sure are we about the mean". Different questions, both called 'a band' — captions must say which.

## Wrap-up → Block 3

Pipeline: **resample → ACF → matched window → decompose → band.** Your series next.